# Milestone 3–4 — 강화학습으로 천 두 번 접기 (Colab 버전)

로컬 PC와 같은 코드(`rl/train_ppo.py`, `rl/eval_policy.py`)를 Colab에서 돌립니다.

- **한 RL step = 로봇의 pick-and-place 한 번(접기 1회)**, 에피소드 = 2 step.
- 행동(기본): `MultiDiscrete([25, 25])` — 천 키포인트 *i* 를 집어 키포인트 *j* 위에 놓기. (`--action xy`는 연속 좌표, 더 어려움)
- 관측과 보상은 여러분이 `rl/features.py`, `rl/reward.py`에 작성합니다 (먼저 로컬에서 `python rl/check_rl.py` 통과).
- 시뮬레이터는 **CPU**로 돕니다 (GPU는 거의 도움이 안 됨). 무료 Colab은 CPU 2개 → `--n_envs 2`. 접기 1회 ≈ 1–2초.

| 설정 | 접기 수 | 시간 (대략) |
|---|---|---|
| Colab 무료, `--n_envs 2 --steps 1000` | 1000 | 30–45분 |
| 로컬 8코어, `--n_envs 8 --steps 4000` | 4000 | 45–60분 |

세션이 끊기면 `--resume runs/m3/ppo/ckpt_<N>.zip` 으로 이어서 학습할 수 있습니다. Google Drive에 저장하려면 6번 셀을 쓰세요.

In [ ]:
# 0. 저장소 준비: Colab이면 GitHub에서 받고, 로컬(Jupyter)이면 이 노트북이 있는 저장소를 그대로 씁니다.
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/<org>/DL_assignment02.git'   # TODO: 수업 저장소 주소 (또는 아래에서 zip 업로드)
if IN_COLAB:
    if not os.path.exists('/content/DL_assignment02'):
        if '<org>' in REPO_URL:   # 주소가 없으면 저장소 zip을 직접 올립니다
            from google.colab import files
            up = files.upload(); z = next(iter(up)); subprocess.run(['unzip', '-q', z, '-d', '/content'], check=True)
            top = [d for d in os.listdir('/content') if d.startswith('DL_assignment02')][0]; os.rename(f'/content/{top}', '/content/DL_assignment02')
        else:
            subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/DL_assignment02'], check=True)
    os.chdir('/content/DL_assignment02')
else:
    os.chdir('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
print(os.getcwd(), os.listdir('.'))

In [ ]:
# 1. 패키지 (MuJoCo는 3.3.1로 고정: 새 버전은 천-테이블 접촉이 50개로 제한되어 천이 테이블을 뚫습니다)
if IN_COLAB:
    os.environ['MUJOCO_GL'] = 'egl'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mujoco==3.3.1', 'robosuite==1.5.2', 'gymnasium', 'stable-baselines3', 'imageio-ffmpeg', 'opencv-python-headless'], check=True)
TEAM = 'T01'                     # TODO: 팀 ID
MESH = ''                        # 내 천: 'assets/T01_cloth' (Milestone 1에서 만든 폴더를 올려 두세요). 비우면 예시 천
N_ENVS = os.cpu_count()          # Colab 무료 = 2
STEPS = 1000                     # 학습할 접기 수 (에피소드 = 2 접기)
mesh_arg = f'--mesh {MESH}' if MESH else ''
print('CPUs', N_ENVS)

In [ ]:
# 2. 설치 확인 (Milestone 0)
!"{sys.executable}" scripts/check_env.py --student {TEAM}

In [ ]:
# 3. 내 features.py / reward.py 확인 (Milestone 3). 모두 PASS 여야 학습합니다.
#    Colab에서 직접 고치려면 왼쪽 파일 탭에서 rl/features.py, rl/reward.py를 열어 편집하세요.
!"{sys.executable}" rl/check_rl.py

In [ ]:
# 4. 학습 (PPO). 진행 상황: 최근 50 에피소드의 평균 return과 성공률
!"{sys.executable}" rl/train_ppo.py --student {TEAM} {mesh_arg} --n_envs {N_ENVS} --steps {STEPS} --n_steps 8 --batch 32 --save_every 200

In [ ]:
# 5. 학습 곡선
import matplotlib.pyplot as plt, matplotlib.image as mpimg
plt.figure(figsize=(12, 4)); plt.imshow(mpimg.imread('runs/m3/ppo/curve.png')); plt.axis('off'); plt.show()

In [ ]:
# 6. (선택) Google Drive에 결과 저장 / 다음 세션에서 이어 학습할 때
SAVE_TO_DRIVE = False
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    import shutil; shutil.copytree('runs', f'/content/drive/MyDrive/DL_assignment02_{TEAM}_runs', dirs_exist_ok=True)

In [ ]:
# 7. 평가 (Milestone 4): 공통 평가 시드에서 성공률. 기준선(random)과 비교합니다.
#    시간을 줄이려면 --n 20. 제출용은 50개 전부.
!"{sys.executable}" rl/eval_policy.py --agent random --n 20 --workers {N_ENVS} {mesh_arg}
!"{sys.executable}" rl/eval_policy.py --agent ppo --ckpt runs/m3/ppo/ckpt.zip --n 20 --workers {N_ENVS} --video 2 {mesh_arg}

In [ ]:
# 8. 결과 영상 보기
from IPython.display import Video, display
import glob
for f in sorted(glob.glob('runs/m4/ppo/videos/*.mp4'))[:2]: display(Video(f, embed=True, width=640))

In [ ]:
# 9. 결과 내려받기 (로컬 저장소의 runs/ 에 풀어 넣고 make_submission.py로 제출 zip을 만드세요)
if IN_COLAB:
    import shutil; from google.colab import files
    shutil.make_archive(f'runs_{TEAM}', 'zip', '.', 'runs'); files.download(f'runs_{TEAM}.zip')